# Transfer Trumans To AMASS Format

In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np


base_path  = "./data/LINGO/dataset"  
output_dir = "./data/LINGO_Motion_Processed/npz_y_up"
os.makedirs(output_dir, exist_ok=True)


END_IS_INCLUSIVE = True

# =========================
# Load globally concatenated data
# =========================
human_pose   = np.load(os.path.join(base_path, "human_pose.npy"),   mmap_mode="r")  # (N, 63)
human_orient = np.load(os.path.join(base_path, "human_orient.npy"), mmap_mode="r")  # (N, 3)
transl       = np.load(os.path.join(base_path, "transl_aligned.npy"), mmap_mode="r")# (N, 3)

# Start and end indices for each segment (M x 3 or M)
start_idx = np.load(os.path.join(base_path, "start_idx.npy"))
end_idx   = np.load(os.path.join(base_path, "end_idx.npy"))

# Optional: scene name for each frame and text for each segment
scene_name_per_frame = np.load(os.path.join(base_path, "scene_name.pkl"), allow_pickle=True)  # (N,)
text_aug = np.load(os.path.join(base_path, "text_aug.pkl"), allow_pickle=True)                # (M,)

# =========================
# Parse start/end indices (supports Mx3 / M / Mx1)
# =========================
def parse_idx(arr):
    arr = np.asarray(arr)
    if arr.ndim == 1:
        idx = arr
    elif arr.ndim == 2:
        idx = arr[:, -1]
    else:
        raise ValueError(f"Unexpected idx shape: {arr.shape}")
    return idx.astype(np.int64)

start_f = parse_idx(start_idx)
end_f   = parse_idx(end_idx)

M = len(start_f)
print("Total number of motion segments:", M)
print("Total number of global frames N:", human_pose.shape[0])

# Fixed frame rate (modify it if specified differently by the dataset)
mocap_framerate = 30

# =========================
# Save each segment
# =========================
for seg_i in range(M):
    s = int(start_f[seg_i])
    e = int(end_f[seg_i])

    if END_IS_INCLUSIVE:
        e_excl = e + 1
    else:
        e_excl = e

    # Validity check
    if s < 0 or e_excl <= s or e_excl > human_pose.shape[0]:
        print(f"[Skip] seg={seg_i:06d}, s={s}, e={e}, e_excl={e_excl}")
        continue

    # Slice frame data for the current segment
    transl_seg = transl[s:e_excl]         # (T, 3)
    orient_seg = human_orient[s:e_excl]   # (T, 3)
    pose_seg   = human_pose[s:e_excl]     # (T, 63)

    T = pose_seg.shape[0]

    # LINGO does not provide hand poses, so fill them with zeros (T, 45)
    lhand = np.zeros((T, 45), dtype=pose_seg.dtype)
    rhand = np.zeros((T, 45), dtype=pose_seg.dtype)

    # Keep the output pose dimension consistent with the original script:
    # (T, 156) = orient(3) + body(63) + left hand(45) + right hand(45)
    poses = np.concatenate([orient_seg, pose_seg, lhand, rhand], axis=1)

    # Segment-level metadata: text and scene name.
    # Use the first frame's scene name, or replace it with the mode if needed.
    text = text_aug[seg_i] if seg_i < len(text_aug) else ""
    scene = scene_name_per_frame[s] if scene_name_per_frame is not None else ""

    out_file = os.path.join(output_dir, f"{seg_i:06d}.npz")
    np.savez(
        out_file,
        trans=transl_seg,
        poses=poses,
        mocap_framerate=mocap_framerate,
        text=text,
        scene=scene,
        start_idx=s,
        end_idx=e,
        y_up=True,
    )

    if seg_i % 50 == 0 or seg_i == M - 1:
        print(f"[{seg_i+1}/{M}] saved {out_file}  T={T}")

print("✅ All files have been saved. Output directory:", output_dir)

# Transfer Y-UP To Z-UP

In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
from scipy.spatial.transform import Rotation as R

IN_DIR  = "./data/LINGO_Motion_Processed/npz_y_up"
OUT_DIR = "./data/LINGO_Motion_Processed/npz_z_up"
os.makedirs(OUT_DIR, exist_ok=True)

# Y-up -> Z-up: active rotation of +90 degrees around the X-axis
# Mapping: (x, y, z) -> (x, -z, y)
R_YUP_TO_ZUP = np.array([
    [1, 0,  0],
    [0, 0, -1],
    [0, 1,  0],
], dtype=np.float32)

def _get(npz, key, default):
    return npz[key] if key in npz.files else default

def yup_to_zup_trans(trans):
    trans = np.asarray(trans, dtype=np.float32)
    return (trans @ R_YUP_TO_ZUP.T) + np.array([0.0, +0.2, -0.4], dtype=np.float32)

def yup_to_zup_root_orient_axis_angle(orient_axis_angle):
    aa = np.asarray(orient_axis_angle, dtype=np.float32).reshape(-1, 3)  # (T,3)
    rotm = R.from_rotvec(aa).as_matrix().astype(np.float32)              # (T,3,3)
    # p_z = A p_y  =>  R_z = A R_y   (R represents body-to-world rotation)
    rotm2 = (R_YUP_TO_ZUP[None, :, :] @ rotm).astype(np.float32)         # (T,3,3)
    aa2 = R.from_matrix(rotm2).as_rotvec().astype(np.float32)
    return aa2.reshape(orient_axis_angle.shape).astype(np.float32)

def _to_py_scalar(x):
    # Support NumPy scalars and zero-dimensional arrays
    if isinstance(x, np.ndarray) and x.shape == ():
        return x.item()
    return x

def process_file(in_file, out_file):
    data = np.load(in_file, allow_pickle=True)

    trans = data["trans"]
    poses = data["poses"]
    assert poses.ndim == 2 and poses.shape[1] >= 3, f"Invalid poses shape: {poses.shape}"

    # LINGO compatibility: use default values when gender or betas are unavailable
    gender = _to_py_scalar(_get(data, "gender", "neutral"))
    betas  = _get(data, "betas", np.zeros((10,), dtype=np.float32))  # Commonly uses 10 dimensions
    mocap_framerate = _to_py_scalar(_get(data, "mocap_framerate", 30))

    # Preserve optional fields when available
    text = _to_py_scalar(_get(data, "text", ""))
    scene = _to_py_scalar(_get(data, "scene", ""))
    start_idx = _to_py_scalar(_get(data, "start_idx", -1))
    end_idx = _to_py_scalar(_get(data, "end_idx", -1))
    y_up = _to_py_scalar(_get(data, "y_up", True))

    # Coordinate transformation
    trans_zup = yup_to_zup_trans(trans)
    orient_yup = poses[:, :3]
    orient_zup = yup_to_zup_root_orient_axis_angle(orient_yup)

    poses_zup = np.asarray(poses, dtype=np.float32).copy()
    poses_zup[:, :3] = orient_zup

    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    np.savez(
        out_file,
        trans=trans_zup.astype(np.float32),
        poses=poses_zup.astype(np.float32),
        gender=gender,
        mocap_framerate=int(mocap_framerate),
        betas=np.asarray(betas, dtype=np.float32),

        # Preserve additional LINGO information
        text=text,
        scene=scene,
        start_idx=int(start_idx) if start_idx != -1 else -1,
        end_idx=int(end_idx) if end_idx != -1 else -1,
        y_up=bool(y_up),
        z_up=True,
        offset=offset.copy(),
    )

def main():
    files = sorted([f for f in os.listdir(IN_DIR) if f.endswith(".npz")])
    print(f"Found {len(files)} files")

    for i, fname in enumerate(files, 1):
        in_file  = os.path.join(IN_DIR, fname)
        out_file = os.path.join(OUT_DIR, fname)
        process_file(in_file, out_file)

        if i % 50 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] done: {fname}")

    print("All files have been converted. Output directory:", OUT_DIR)

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
import os
import glob
import shutil
import numpy as np

SRC_DIR = "./data/LINGO_Motion_Processed/npz_z_up"
DST_DIR = "./data/LINGO_Motion_Processed/npz_z_up_inference"
TH = 120

os.makedirs(DST_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(SRC_DIR, "*.npz")))
print("Number of source files:", len(files))

keep = 0
for i, f in enumerate(files, 1):
    data = np.load(f, allow_pickle=True)
    if "poses" not in data.files:
        continue
    T = data["poses"].shape[0]
    if T > TH:
        shutil.copy2(f, os.path.join(DST_DIR, os.path.basename(f)))
        keep += 1

    if i % 200 == 0 or i == len(files):
        print(f"[{i}/{len(files)}] keep={keep}")

print("Completed")
print("Number of files with T > 121:", keep)
print("Output directory:", DST_DIR)

# Convert to PKL

In [ ]:
!python ./scripts/data_process/convert_amass_data.py \
  --path ./data/LINGO_Motion_Processed/npz_z_up_inference \
  --debug \
  --upright_start\
  --save_path ./data/lingo_motion_inference.pkl